# Metin Temsili

- Bir metni sayısal veya başka türde bir formatta temsil etme işlemidir.

## Table of Contents

1. [Bag of Words](#1.-Bag-of-Words)
    - [IMDB Data Set](#1.a.-IMDB-Data-Set)
2. [TF-IDF (Term Frequency-Inverse Document Frequency)](#2.-TF-IDF-(Term-Frequency-Inverse-Document-Frequency))

## 1. Bag of Words
- NLP ve metin madenciliğinde kullanılan temel bir metin temsili yöntemidir.
- Metinlerdeki kelimeleri sayısal verilere dönüştürür ve metinlerin analizini sağlar.
- İşleyişi:
    - Kelime kümesi oluşturma.
    - Kelime frekansı hesaplama.
    - Vektör temsili.

In [2]:
from sklearn.feature_extraction.text import CountVectorizer

In [2]:
documents = ["kedi bahçede", "kedi evde"]

vectorizer = CountVectorizer()

X = vectorizer.fit_transform(documents)

In [4]:
print(X)

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 4 stored elements and shape (2, 3)>
  Coords	Values
  (0, 2)	1
  (0, 0)	1
  (1, 2)	1
  (1, 1)	1


In [6]:
# Kelime kümesi oluşturma
feature_names = vectorizer.get_feature_names_out()
vector_temsili = X.toarray()

vector_temsili

array([[1, 0, 1],
       [0, 1, 1]])

In [7]:
feature_names

array(['bahçede', 'evde', 'kedi'], dtype=object)

|**Kelimeler**|*bahçede*|*evde*|*kedi*
|---------|----------|---------|----|
|**1. Cümle**|1|0|1|
|**2. Cümle**|0|1|1|

### 1.a. IMDB Data Set

In [44]:
import pandas as pd
import re
from collections import Counter
from nltk.corpus import stopwords
from bs4 import BeautifulSoup as bs

df = pd.read_csv("IMDB Dataset.csv")

In [4]:
documents = df["review"]
labels = df["sentiment"]

In [15]:
stop_words_eng = set(stopwords.words("english"))

In [54]:
def clean_text(text):
    text = text.lower()

    # Rakamları temizlemek için yazdığımız regular expresion.
    text = re.sub(r"\d+", "", text)

    text = bs(text, "html.parser").get_text()
    
     # Özel karakterlerin temizlenmesi için yazdığımız regular expression. '^' = not, '\w' = harfler, '\s' = boşluk
    text = re.sub(r"[^\w\s]", "", text)

    ttokens = text.split()
    text = " ".join([word for word in ttokens if word not in stop_words_eng])

    return text

In [46]:
cleaned_documents = df["review"].apply(clean_text)

In [47]:
vectorizer = CountVectorizer()

X = vectorizer.fit_transform(cleaned_documents[:75])

In [48]:
feature_names = vectorizer.get_feature_names_out()
word_freq = dict(zip(feature_names, X.sum(axis=0).A1))


bow_df = pd.DataFrame.from_dict({"words": list(word_freq.keys()), "values": list(word_freq.values())})

In [53]:
# En sık kullanılan 5 kelime
bow_df.sort_values(by="values", ascending=False).head(5)

,words,values
2276,movie,123
1270,film,98
2441,one,72
2009,like,59
1449,good,38


## 2. TF-IDF (Term Frequency-Inverse Document Frequency)

- Metin madenciliğinde ve bilgi erişiminde sıkça kullanılan bir özellik çıkarım yöntemidir.
- Kelimelerin belgeler içinde ne kadar önemli olduğunu belirlemek için kullanılır.
- **Term Frequency (TF):** Bir kelimenin bir belgede ne kadar sık geçtiğini ölçer.
- **Inverse Document Frequency (IDF):** Bir kelimenin tüm belgelerdeki yaygınlığını ölçer, kelime ne kadar çok belgede geçiyorsa o kelime çok fazla bilgi sağlamaz.

$$ \text{TF}(t,d) = \frac{\text{Sayaç}(t,d)}{\text{Toplam Kelime Sayısı}(d)}$$
<center>Burada, <b><em>Sayaç(t,d)</em></b> denilen kavram <b><em>t</em></b> kelimensinin <b><em>d</em></b> dökümanında geçme sayısıdır. <b><em>Toplam Kelime Sayısı(d)</em></b> ise <b><em>d</em></b> dökümanındaki toplem kelime sayısıdır.</center>

$$ \text{IDF}(t,D) = log(\frac{\text{Toplam Belgeler Sayısı}(D)}{1 + \text{Belgelerde Geçen Sayısı}(t)})$$
<center>Burada <b><em>Toplam Belgeler Sayısı(D)</em></b> toplam döküman sayısı, <b><em>Belgelerde Geçen Sayısı (t)</em></b> ise <b><em>t</em></b> kelimesinin geçtiği belge sayısıdır.</center>

$$ \text{TF-IDF}(t,d,D) = \text{TF}(t,d) \times \text{IDF}(t,D)$$